In [8]:
import os
import cv2

# Set path pointing directly to your local dataset folder
DATASET_DIR = 'dataset'
splits = ['Training', 'Testing']
categories = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']

print("📊 --- Local Split Dataset Audit --- 📊\n")

for split in splits:
    print(f"📁 Checking Split: {split}...")
    split_total = 0
    
    for category in categories:
        folder_path = os.path.join(DATASET_DIR, split, category)
        
        if os.path.exists(folder_path):
            # Read image file extensions
            images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            image_count = len(images)
            split_total += image_count
            print(f"   └── Category '{category}': Found {image_count} images.")
            
            # Simple local check to ensure OpenCV can parse files correctly
            if image_count > 0:
                sample_path = os.path.join(folder_path, images[0])
                sample_img = cv2.imread(sample_path)
                if sample_img is None:
                    print(f"       ⚠️ Warning: OpenCV failed to read sample image: {images[0]}")
        else:
            print(f"   ❌ Folder missing: {folder_path}")
            
    print(f"📈 Total images in {split}: {split_total}\n" + "-"*40 + "\n")

📊 --- Local Split Dataset Audit --- 📊

📁 Checking Split: Training...
   └── Category 'glioma_tumor': Found 826 images.
   └── Category 'meningioma_tumor': Found 822 images.
   └── Category 'no_tumor': Found 395 images.
   └── Category 'pituitary_tumor': Found 827 images.
📈 Total images in Training: 2870
----------------------------------------

📁 Checking Split: Testing...
   └── Category 'glioma_tumor': Found 100 images.
   └── Category 'meningioma_tumor': Found 115 images.
   └── Category 'no_tumor': Found 105 images.
   └── Category 'pituitary_tumor': Found 74 images.
📈 Total images in Testing: 394
----------------------------------------



In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 1. Image Configurations
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 2. Medical Data Augmentation (Medically Sound: No vertical flips)
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator()

# 3. Stream Images directly from Folders
print("🔄 Initializing Data Generators...")
train_generator = train_datagen.flow_from_directory(
    'dataset/Training',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    'dataset/Testing',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# 4. Calculate Class Weights to defeat the Imbalance
classes = train_generator.classes
class_series = np.unique(classes)
weights = compute_class_weight(class_weight='balanced', classes=class_series, y=classes)
class_weights = dict(zip(class_series, weights))

print("\n⚖️ Calculated Class Weights to balance training:")
for idx, class_name in enumerate(train_generator.class_indices.keys()):
    print(f"   └── {class_name}: {class_weights[idx]:.4f}")

🔄 Initializing Data Generators...
Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.

⚖️ Calculated Class Weights to balance training:
   └── glioma_tumor: 0.8686
   └── meningioma_tumor: 0.8729
   └── no_tumor: 1.8165
   └── pituitary_tumor: 0.8676


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB0

# 1. DEFINE DATA AUGMENTATION PIPELINE
# This trains the AI on flips, rotations, and zooms so it recognizes tumors from any scanner or angle!
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),      # Handles head tilts in different MRIs
    tf.keras.layers.RandomZoom(0.15),         # Handles varying zoom levels from different scanners
    tf.keras.layers.RandomContrast(0.15)      # Handles differences in scan machine brightness
])

# 2. BUILD THE ARCHITECTURE
inputs = tf.keras.layers.Input(shape=(224, 224, 3))

# Pass inputs through the data augmentation block (only runs during active training phase)
x = data_augmentation(inputs)

# Load Pre-trained EfficientNetB0 base (CRITICAL: Expects 0-255 inputs, which x provides!)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=x)

# Freeze the base layers to protect the foundational pre-trained feature extractors
base_model.trainable = False

# Build the Custom Medical Head wrapper
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)  # Strong dropout block to stop overfitting completely
outputs = Dense(4, activation='softmax')(x)  # 4 target classes

# Combine into the final production model structure
model = Model(inputs=inputs, outputs=outputs)

# 3. COMPILE THE MODEL
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("🚀 Upgraded EfficientNetB0 Core Architecture with Data Augmentation successfully compiled!")
print(f"Total Model Layers: {len(model.layers)}")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
✅ EfficientNetB0 Medical Architecture successfully compiled!
Total Model Layers: 242


In [ ]:
# 4. Define Smart Early Stopping and Learning Rate Callbacks
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=4, 
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=2, 
    min_lr=1e-6,
    verbose=1
)

# 5. Start the Training Run with Upgraded Architecture
print("🚀 Training started... Sit back, this might take a few minutes!")

history = model.fit(
    train_generator,
    epochs=12,                       # 12 epochs is the sweet spot for data augmentation fine-tuning
    validation_data=test_generator,
    class_weight=class_weights,      # Keeps your data classes completely balanced
    callbacks=[early_stop, reduce_lr]
)

print("\n⚙️ Training Complete! Saving the final model matrix...")

# CRITICAL: Save it directly into backend-api so your server picks it up instantly
model.save('backend-api/brain_tumor_efficientnet.h5')
print("✅ Smart model compiled and saved directly into backend-api folder!")

🏋️‍♂️ Training started... Sit back, this might take a few minutes!
Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 175s 2s/step - accuracy: 0.2387 - loss: 1.5775 - val_accuracy: 0.1878 - val_loss: 1.3975 - learning_rate: 0.0010
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.2557 - loss: 1.5120 - val_accuracy: 0.1878 - val_loss: 1.3977 - learning_rate: 0.0010
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - accuracy: 0.2624 - loss: 1.5085 - val_accuracy: 0.2310 - val_loss: 1.3829 - learning_rate: 0.0010
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.2669 - loss: 1.4781 - val_accuracy: 0.2437 - val_loss: 1.3971 - learning_rate: 0.0010
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2436 - loss: 1.4823
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
90/90 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - accuracy: 0.2693 - loss: 1.4827 - val_accuracy: 0.2284 - val_loss: 1.3963 - learning_rate: 0.0010
Epoch 6/10
90/90 ━━━━━━━━━━━━


🎉 Training Complete! Saving the final model matrix...
💾 Saved as: brain_tumor_efficientnet.h5
